# Template: Launching a Nextflow pipeline

**This notebook is a template.** It ships preconfigured to install Nextflow, wire it up to Google Cloud Batch, and run `nf-core/viralrecon` against a small synthetic test dataset. You'll customize a few variables (see "How to use this template" below) before running it against your own data or switching to a different nf-core pipeline.

**What this notebook does:** installs Nextflow + JDK, writes a `nextflow.config` pointed at GCP Batch, dry-runs the pipeline with `-preview` (cheap sanity check), then launches it on the built-in test dataset. Cells further down build a samplesheet from your project's real data and show a commented-out real-data launch you uncomment when you're ready.

We're using `nf-core/viralrecon` here because it fits paired-end viral whole-genome sequencing, which is what the H1N1 demo data is. To swap in a different pipeline, change `PIPELINE_REPO` and `PIPELINE_REVISION` in the parameter cell — the launch mechanics transfer directly.

**Time estimates:**

- **First run (cold)**: 30-45 minutes. Most of the wall clock is GCP Batch allocating Spot VMs and each VM pulling the container image it needs (viralrecon uses ~15 different containers). Containers get cached on the work bucket after the first pull.
- **Subsequent runs (cached)**: 10-15 minutes. Reusing cached containers is fast; the only delay is Spot VM allocation and the actual computational work.

**Cost**: cents on the `test` profile (synthetic dataset). Real data on real hardware is dollars to hundreds of dollars depending on volume — read "Switching from test to real data" carefully before uncommenting anything.

> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (Local)`** (under "Start python Kernel"). Don't pick `Python 3 (ipykernel) (Local)`, PyTorch, or TensorFlow — those are missing libraries this notebook needs and will fail at the first GCS read with `ModuleNotFoundError`. If you already see `Python 3 (Local)` in the top-right of this tab, you're good to go.

## How to use this template

This notebook has three run modes. Pick the one that matches what you're doing:

**Mode 1 — First run on the test dataset (recommended first pass).**
Change nothing. `Kernel → Restart & Run All`. Runs viralrecon's built-in synthetic dataset via the `test` profile. Wall clock: 30-45 min cold. Cost: cents. Use this to prove the plumbing works end-to-end in your project.

**Mode 2 — Run viralrecon on YOUR data.**

1. Do Mode 1 first and confirm the test run finished cleanly.
2. Run the "Samplesheet builder for your dataset" cell partway down; it auto-detects your analytical-dataset bucket and pairs up the FASTQs into `samplesheet.csv`.
3. Review the generated samplesheet.
4. Uncomment the real-data launch cell in the "Switching from test to real data" section and run it. Wall clock: hours. Cost: real dollars.

**Mode 3 — Run a DIFFERENT nf-core pipeline.**

1. In the parameter cell, change `PIPELINE_REPO` (e.g. `nf-core/sarek`, `nf-core/bactopia`, `nf-core/rnaseq`) and `PIPELINE_REVISION` to a specific tagged release.
2. `Kernel → Restart & Run All`. Everything else — install, config, launch mechanics — is the same. Your pipeline's samplesheet columns and params may differ; check the pipeline's docs.

### Which variables do I set?

The parameter cell (next code cell) groups its variables into two blocks with header comments:

| Block | What to do |
|---|---|
| 1. Auto-detected | Leave blank. The environment-check cell fills these in. Only override if auto-detect picks the wrong value. |
| 2. Pipeline selection | Change for Mode 3 (different pipeline). Leave defaults for Modes 1 and 2. |

### When do I restart the kernel?

- **First time opening this notebook in a session**: `Kernel → Restart & Run All`.
- **Changed a parameter after a full run**: re-run from the parameter cell down. Kernel state carries; no full restart needed.
- **Cell errors with something unexplained** (stale variable, half-installed package, weird state): `Kernel → Restart & Run All`.
- **A Nextflow run failed and you've fixed the underlying issue**: re-run just the launch cell. The `-resume` flag reuses cached work; you don't lose completed steps.

## Tools you'll use

This notebook combines Nextflow (workflow engine), nf-core (pipeline
catalog), and GCP Batch (compute backend) to run viralrecon end-to-end.
If any of those names are new, [05-reference.ipynb](05-reference.ipynb)
has a paragraph on each plus the rationale for the pinned versions
this notebook uses.

## Why viralrecon for this data?

`nf-core/viralrecon` is built for paired-end short-read viral
whole-genome sequencing. The H1N1 demo dataset is exactly that:
Illumina paired-end FASTQs from Influenza A samples (SRR13000260,
261, 262, each with `_1`/`_2` pairs).

Different data (RNA-seq, bacterial genomics, metagenomics) just means
a different pipeline name. The launch mechanics here transfer
directly. Browse [the full nf-core catalog](https://nf-co.re/pipelines)
to pick something else.

In [ ]:
import subprocess

DETECTED_PROJECT = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# ============================================================================
# BLOCK 1 — Auto-detected. Leave blank; the next cell fills these in.
# ============================================================================
# Override any of these ONLY if auto-detect picks the wrong value (wrong
# bucket, no default VPC visible, etc.). Values you paste here take
# precedence over auto-detection.

PROJECT_ID     = DETECTED_PROJECT                              # @param {type:"string"}
REGION         = "us-central1"                                 # @param {type:"string"}
WORK_BUCKET    = ""                                            # @param {type:"string"}
RESULTS_BUCKET = ""                                            # @param {type:"string"}
NETWORK        = ""                                            # @param {type:"string"}
SUBNETWORK     = ""                                            # @param {type:"string"}

# ============================================================================
# BLOCK 2 — Pipeline selection. Change for Mode 3 (different pipeline).
# ============================================================================
# Defaults run nf-core/viralrecon 2.6.0 with the `test` profile against
# viralrecon's built-in synthetic dataset. See "Why pinned versions
# matter" in 05-reference.ipynb for the rationale on each pin.
#
# To swap pipelines:
#   1. Set PIPELINE_REPO to the nf-core repo (e.g. "nf-core/sarek",
#      "nf-core/bactopia", "nf-core/rnaseq").
#   2. Set PIPELINE_REVISION to a specific tagged release (avoid "main"
#      — the pipeline's main branch sometimes references config files
#      that haven't been shipped yet).
#   3. Set PROFILE — most nf-core pipelines have a `test` profile too.
#   4. Real-data samplesheet columns may differ; check the pipeline's docs.

PIPELINE_REPO     = "nf-core/viralrecon"                       # @param {type:"string"}
PIPELINE_REVISION = "2.6.0"                                    # @param {type:"string"}
PROFILE           = "test,gcb"                                 # @param {type:"string"}
# `test` = nf-core's built-in tiny synthetic dataset
# `gcb`  = the GCP Batch profile we'll write further down

In [ ]:
import gcsfs
import subprocess

active_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()
print(f"GCP project: {PROJECT_ID}")
print(f"Region:      {REGION}")
print(f"Identity:    {active_account}\n")

fs = gcsfs.GCSFileSystem()
all_buckets = fs.ls("")

if not RESULTS_BUCKET:
    out = [b.rstrip("/") for b in all_buckets if "seqera-output" in b]
    if len(out) > 1:
        print(f"⚠ Multiple seqera-output buckets visible ({len(out)}): {out}")
        print(f"  Auto-selecting the first one. Override RESULTS_BUCKET above if that's wrong.")
    if out:
        RESULTS_BUCKET = f"gs://{out[0]}/"
        print(f"Auto-selected RESULTS_BUCKET: {RESULTS_BUCKET}")
if not WORK_BUCKET:
    # Use the same bucket with a /nf-work/ prefix to keep work staging separate
    WORK_BUCKET = RESULTS_BUCKET.rstrip("/") + "/nf-work/"
    print(f"Auto-selected WORK_BUCKET:    {WORK_BUCKET}")

# Auto-detect the project's custom VPC. APGAP projects don't have the
# default GCP VPC; they use one custom network per project (e.g.
# `h1n1-network-rt8r`). The service account can list SUBNETS but typically
# not NETWORKS (missing compute.networks.list), so query subnets first and
# extract the parent network from each row's network field. Only fall back
# to a networks listing if the subnet query returned nothing AND NETWORK
# wasn't already set in the parameter cell. Pattern courtesy of Glen
# Otero's launch notebook.
#
# Collect stderr from each empty-result query so the RuntimeError below
# can surface *why* auto-detect came back empty (permission denied vs.
# no VPC in the project vs. wrong region), instead of a silent (none).
def _detect_network_subnet(network, subnet):
    diagnostics = []
    if not subnet:
        r = subprocess.run(
            ["gcloud", "compute", "networks", "subnets", "list",
             f"--project={PROJECT_ID}",
             f"--regions={REGION}",
             "--format=value(name,network)",
             "--limit=1"],
            capture_output=True, text=True,
        )
        rows = r.stdout.strip().splitlines()
        if rows:
            parts = rows[0].split()
            subnet = parts[0]
            if not network and len(parts) > 1:
                # network field is a full URL; bare name is the last segment.
                network = parts[1].rsplit("/", 1)[-1]
        elif r.stderr.strip():
            diagnostics.append(f"[subnets list stderr]\n{r.stderr.strip()}")
    if not network:
        r = subprocess.run(
            ["gcloud", "compute", "networks", "list",
             f"--project={PROJECT_ID}",
             "--format=value(name)",
             "--limit=1"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            network = r.stdout.strip().splitlines()[0]
        elif r.stderr.strip():
            diagnostics.append(f"[networks list stderr]\n{r.stderr.strip()}")
    return network, subnet, diagnostics

NETWORK, SUBNETWORK, _detect_diagnostics = _detect_network_subnet(NETWORK, SUBNETWORK)
print(f"Network:      {NETWORK or '(none)'}")
print(f"Subnetwork:   {SUBNETWORK or '(none)'}")

if not NETWORK or not SUBNETWORK:
    diag_block = ("\n\n" + "\n\n".join(_detect_diagnostics)) if _detect_diagnostics else ""
    raise RuntimeError(
        "Couldn't auto-detect a VPC + subnet, and NETWORK or SUBNETWORK is "
        "empty in the parameter cell above. Ask your project lead for the "
        "network and subnet names for this project, then paste them into "
        "NETWORK and SUBNETWORK above and re-run. "
        f"(Got NETWORK={NETWORK!r}, SUBNETWORK={SUBNETWORK!r}.)" + diag_block
    )

## Installing Nextflow and Java

Nextflow runs on the JVM. We install both via whichever conda-compatible package manager the Workbench has on PATH (`micromamba`, `conda`, or `mamba` — the image varies). Versions are pinned (Nextflow 23.10 LTS, JDK 17, viralrecon 2.6.0); see [05-reference.ipynb](05-reference.ipynb) for what each pin is working around.

Where the install lands depends on the package manager:

- **`micromamba`**: installs into a per-user env at `~/nf-env`, with its own root prefix at `~/.micromamba`. Some Workbench images make the shared `/opt/micromamba/pkgs/cache` read-only for the `jupyter` user, which breaks a system-wide `-n base` install. A per-user env sidesteps that — everything writes under `$HOME`.
- **`conda` / `mamba`**: installs into the active env (typically `base`).

Either way, the cell is a no-op if the pinned versions are already there. If the install fails (network glitch, bioconda channel hiccup), just re-run the cell.

In [ ]:
import os
import shutil
import subprocess

# Workbench images vary: newer ones ship `micromamba`, older ones `conda`.
# `mamba` is sometimes a thin wrapper around one of the others and can
# behave unpredictably with conda-style flags, so it sits last in the
# preference order.
for _mgr in ("micromamba", "conda", "mamba"):
    if shutil.which(_mgr):
        PKG_MANAGER = _mgr
        break
else:
    raise RuntimeError(
        "No conda-compatible package manager (micromamba, conda, mamba) "
        "found in PATH. This notebook needs one to install Nextflow + JDK "
        "from the bioconda / conda-forge channels."
    )

# Install path differs between package managers:
#
# - `micromamba`: install into a per-user env at ~/nf-env with its own
#   root-prefix at ~/.micromamba. Some Workbench images lock down
#   /opt/micromamba/pkgs/cache for the jupyter user, which breaks
#   `micromamba install -n base`. A per-user env writes everything
#   under $HOME.
# - `conda` / `mamba`: install into the active env (typically base).
if PKG_MANAGER == "micromamba":
    NF_ENV = os.path.expanduser("~/nf-env")
    MAMBA_ROOT = os.path.expanduser("~/.micromamba")
    verb = "install" if os.path.exists(os.path.join(NF_ENV, "conda-meta")) else "create"
    _install_cmd = [
        "micromamba", verb, "-y",
        "--prefix", NF_ENV,
        "--root-prefix", MAMBA_ROOT,
    ]
else:
    _install_cmd = [PKG_MANAGER, "install", "-y"]

_install_cmd += ["-c", "bioconda", "-c", "conda-forge",
                 "openjdk=17", "nextflow=23.10"]

print(f"Pinning Nextflow 23.10 + JDK 17 via {PKG_MANAGER}...")
subprocess.run(_install_cmd, check=True)

# For micromamba's per-user env, add the env's bin dir to PATH so the
# next cell (and everything after) finds the nextflow + java binaries.
# conda/mamba install into the active env which is already on PATH.
if PKG_MANAGER == "micromamba":
    env_bin = os.path.join(NF_ENV, "bin")
    if env_bin not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = env_bin + os.pathsep + os.environ["PATH"]
    print(f"Added {env_bin} to PATH")

print("Install complete.")

In [ ]:
import subprocess

print("=== java -version ===")
subprocess.run(["java", "-version"], check=True)
print("\n=== nextflow -version ===")
subprocess.run(["nextflow", "-version"], check=True)

## Pointing Nextflow at GCP Batch

Nextflow needs four pieces of information:

- The executor to use. `google-batch` submits each pipeline step as a
  GCP Batch job.
- The GCP project and region to run jobs in.
- A staging bucket for intermediate work. Nextflow writes per-step
  inputs and outputs here as the pipeline runs.
- A service account. We default to the Workbench's own service
  account, which Seqera uses for this project too, so the permissions
  are already in place.

All of this goes into a `nextflow.config` file. The next cell writes
it; the one after that prints it back out so you can see what landed
on disk.

In [ ]:
config_content = f"""
profiles {{
    gcb {{
        // Merge ALL process-scope settings into ONE block.
        // Never mix dot-syntax (process.executor = ...) with a block-syntax
        // process {{ ... }} in the same profile — Nextflow's legacy parser
        // silently discards the dot-syntax assignment when it then sees a
        // block. The first time we hit this, the executor silently fell
        // back to 'local' and the pipeline failed on the GCS work directory.
        process {{
            executor      = 'google-batch'
            cpus          = 2
            memory        = '4 GB'
            // Retry transient Batch failures (VM preemption, brief network
            // hiccups) up to twice before giving up on the task.
            errorStrategy = {{ task.attempt <= 2 ? 'retry' : 'finish' }}
            maxRetries    = 2
        }}

        google {{
            project  = '{PROJECT_ID}'
            location = '{REGION}'
            batch {{
                serviceAccountEmail = '{active_account}'
                // APGAP projects use a custom VPC rather than the default
                // network. Batch needs to know which network and subnet
                // to attach VMs to. Auto-detected above; override the
                // NETWORK / SUBNETWORK params if the wrong ones are picked.
                network    = 'projects/{PROJECT_ID}/global/networks/{NETWORK}'
                subnetwork = 'projects/{PROJECT_ID}/regions/{REGION}/subnetworks/{SUBNETWORK}'
                // Spot VMs cut cost ~70% but can be preempted. The
                // errorStrategy retry above handles the occasional
                // preemption. Set to false if you need guaranteed
                // completion (e.g., overnight production runs).
                spot = true
            }}
        }}

        // Where Nextflow stages intermediate files.
        workDir = '{WORK_BUCKET}'
    }}
}}
"""

with open("nextflow.config", "w") as f:
    f.write(config_content)

print("Wrote nextflow.config")

In [ ]:
with open("nextflow.config") as f:
    print(f.read())

## Step 1: dry run with `-preview`

Before sending anything to GCP Batch (which costs real money), ask
Nextflow to parse the pipeline and our config without actually running
steps. If there's a syntax error in the config or Nextflow can't
resolve the pipeline, we find out in ten seconds instead of ten
minutes.

In [ ]:
import subprocess

result = subprocess.run(
    ["nextflow", "run", PIPELINE_REPO,
     "-r", PIPELINE_REVISION,
     "-profile", PROFILE,
     "-preview",
     "-c", "nextflow.config",
     # No trailing slash: Nextflow appends its own / when joining the
     # outdir with per-step relative paths, and a trailing / here produces
     # a malformed `gs://bucket/path//step/file` that breaks publishFile.
     "--outdir", RESULTS_BUCKET.rstrip("/") + "/viralrecon-test-results"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("DRY RUN FAILED — fix config before continuing:")
    print(result.stderr)

## Step 2: actually run the test pipeline

Runs `nf-core/viralrecon` on the synthetic test dataset bundled with
the `test` profile. Each pipeline step becomes a GCP Batch job. As
steps complete, you'll see Nextflow's progress table update inline.

Expected duration: 30-45 minutes cold, 10-15 minutes cached (per the
"Time estimates" section near the top). The test profile's actual
compute is only a few minutes; the rest is GCP Batch scheduling
overhead — Spot VM allocation and per-VM container pulls.

Expected cost: a few cents.

If the cell still shows "running" several minutes after Nextflow's
final log line (`Pipeline completed successfully`), interrupt the
kernel and continue to the next cells. There's a known interaction
with Nextflow's background Batch-polling threads that occasionally
keeps the subprocess pipe from cleanly signaling completion to
Jupyter.

Want to watch what's happening on GCP? Open
https://console.cloud.google.com/batch/jobs in a new tab while it
runs.

In [ ]:
import select
import subprocess

# We use `-resume` so if you re-run this cell, Nextflow will reuse cached
# results from any steps that already completed.
process = subprocess.Popen(
    # `-ansi-log false` disables Nextflow's in-place ANSI refresh of the
    # process table. Jupyter cell output can't interpret ANSI redraws and
    # otherwise renders the same table repeatedly as hundreds of appended
    # `[-]` marker lines, which makes a healthy run look stuck. With this
    # flag Nextflow prints one line per event (submitted, terminated, etc.)
    # and the completion message is at the bottom where users expect it.
    ["nextflow", "-ansi-log", "false",
     "run", PIPELINE_REPO,
     "-r", PIPELINE_REVISION,
     "-profile", PROFILE,
     "-c", "nextflow.config",
     "-resume",
     # No trailing slash: see comment in the dry-run cell above.
     "--outdir", RESULTS_BUCKET.rstrip("/") + "/viralrecon-test-results"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

# Stream stdout. Nextflow's JVM spawns Batch-polling threads that hold
# the stdout pipe open even after the main process exits, so a plain
# `for line in process.stdout` or a final .read() drain would hang
# waiting for an EOF that never arrives. We use select() with a 1-second
# timeout to read only when data is ready, and we stop when select sees
# no buffered data AND the main process has exited (poll() returns a
# code instead of None).
try:
    while True:
        rlist, _, _ = select.select([process.stdout], [], [], 1.0)
        if rlist:
            line = process.stdout.readline()
            if line:
                print(line, end="", flush=True)
        elif process.poll() is not None:
            # No data buffered and the main process has exited.
            break
finally:
    process.stdout.close()

print(f"\nNextflow exited with code {process.returncode}")

## What just happened

Nextflow pulled `nf-core/viralrecon` from GitHub. (Subsequent runs
reuse the cached copy.)

For each step in the pipeline it submitted a GCP Batch job, waited
for the job to finish, and pulled the outputs back.

Intermediate files went into `{WORK_BUCKET}`. Usually safe to delete
after a successful run; see the cleanup section below.

Final outputs landed in the results bucket from `--outdir`. For a map
of what viralrecon writes where, see [05-reference.ipynb](05-reference.ipynb).

The full Nextflow log is in `.nextflow.log` in your notebook
directory. An HTML execution report sits in `<results>/pipeline_info/`.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
# Mirror the no-trailing-slash convention used in --outdir on the launch
# command so listing matches what Nextflow actually wrote.
results_prefix = RESULTS_BUCKET.replace("gs://", "").rstrip("/") + "/viralrecon-test-results"

try:
    entries = fs.ls(results_prefix)
    print(f"Contents of gs://{results_prefix}/ ({len(entries)} items):")
    for e in sorted(entries):
        info = fs.info(e)
        kind = "DIR " if info.get("type") == "directory" else "FILE"
        size_mb = info.get("size", 0) / 1024 / 1024
        print(f"  [{kind}] {size_mb:>7.1f} MB  {e.rsplit('/', 1)[-1]}")
except FileNotFoundError:
    print("Results bucket prefix not found. Did the pipeline run complete successfully?")

## The MultiQC report

`nf-core/viralrecon` includes a MultiQC step that aggregates QC
metrics from every other step into a single self-contained HTML
report. That's the first thing to look at after a run finishes.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
# MultiQC report location depends on the pipeline; viralrecon puts it under
# multiqc/.../multiqc_report.html
multiqc_candidates = [
    p for p in fs.find(results_prefix)
    if p.endswith("multiqc_report.html")
]

if multiqc_candidates:
    report_path = multiqc_candidates[0]
    print(f"MultiQC report: gs://{report_path}")
    print(f"\nTo view: open `gs://{report_path}` in the JupyterLab GCS browser")
    print("(left sidebar), right-click → Download, then open the local copy.")
else:
    print("No multiqc_report.html found yet — pipeline may still be running.")

## Variant calls and consensus sequences

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()

vcfs = [p for p in fs.find(results_prefix) if p.endswith((".vcf", ".vcf.gz"))]
consensus = [p for p in fs.find(results_prefix) if p.endswith(".consensus.fa")]

def _show(label, paths, limit=5):
    print(f"{label} ({len(paths)}):")
    for p in paths[:limit]:
        size_kb = fs.info(p).get("size", 0) / 1024
        print(f"  {size_kb:>7.1f} KB  gs://{p}")
    if len(paths) > limit:
        print(f"  ... and {len(paths) - limit} more")

_show("VCF files", vcfs)
print()
_show("Consensus FASTA files", consensus)

## Samplesheet builder for your dataset

The next cell scans your project's analytical-dataset bucket, pairs up the `_1.fastq.gz` / `_2.fastq.gz` files, and writes a viralrecon-format `samplesheet.csv` to the notebook directory. That's the input the real-data launch cell (further down) expects.

If you're just doing the Mode 1 test run, you can still run this cell — it doesn't launch anything, it just builds the samplesheet for later. For the samplesheet format reference, see [05-reference.ipynb](05-reference.ipynb).

In [ ]:
import gcsfs
import re
import pandas as pd

# Use the same DATASET_URI auto-detection as notebook 02.
DATASET_URI = ""  # @param {type:"string"}  -- paste a dataset URI here to override

fs = gcsfs.GCSFileSystem()
if not DATASET_URI:
    all_analytical = [b.rstrip("/") for b in fs.ls("") if "analytical-dataset" in b]
    pathogen = PROJECT_ID.split("-")[0]
    matched = [b for b in all_analytical if b.startswith(pathogen)]
    candidates = matched or all_analytical
    if not candidates:
        raise RuntimeError("No analytical-dataset bucket visible — set DATASET_URI manually.")
    DATASET_URI = f"gs://{candidates[0]}/"
    print(f"Auto-selected dataset for samplesheet: {DATASET_URI}")

dataset_path = DATASET_URI.replace("gs://", "").rstrip("/")
all_files = sorted(fs.ls(dataset_path))
fastqs = [f for f in all_files if f.endswith(".fastq.gz")]

# Group _1/_2 pairs by sample stem.
pairs = {}
for f in fastqs:
    m = re.search(r"/([^/]+)_([12])\.fastq\.gz$", f)
    if m:
        stem, direction = m.group(1), m.group(2)
        pairs.setdefault(stem, {})[direction] = f"gs://{f}"

# Build the viralrecon samplesheet format:
# sample,fastq_1,fastq_2
rows = []
incomplete = []
for sample, files in sorted(pairs.items()):
    if "1" in files and "2" in files:
        rows.append({
            "sample": sample,
            "fastq_1": files["1"],
            "fastq_2": files["2"],
        })
    else:
        missing = "_2" if "1" in files else "_1"
        incomplete.append(f"{sample} (missing {missing})")

if incomplete:
    print(f"⚠ Skipped {len(incomplete)} sample(s) missing a paired read:")
    for name in incomplete:
        print(f"  - {name}")
    print()

samplesheet = pd.DataFrame(rows)
samplesheet.to_csv("samplesheet.csv", index=False)
print(f"Wrote samplesheet.csv with {len(samplesheet)} samples:\n")
print(samplesheet.to_string(index=False))

### Optional: enrich the samplesheet with APGAP metadata

The samplesheet we just wrote has the three columns viralrecon
expects: `sample`, `fastq_1`, `fastq_2`. The next cell loads APGAP's
`_apgap_metadata.csv` if it's in the dataset bucket, and shows you
the same samples with pathogen, collection facility, instrument, and
so on joined on.

Important: the enriched view is for your reference only. The actual
`samplesheet.csv` we hand to Nextflow stays as the minimal three
columns. viralrecon would ignore the extras anyway.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
metadata_path = DATASET_URI.replace("gs://", "").rstrip("/") + "/_apgap_metadata.csv"

try:
    with fs.open(metadata_path, "r") as f:
        metadata = pd.read_csv(f)
    # APGAP metadata is per-FILE (R1 and R2 are separate rows). For an
    # enriched samplesheet view we collapse to per-sample by picking the R1
    # row's values; sample-level fields (collection date, lab, etc.) are
    # the same for both directions anyway.
    metadata["sample_stem"] = metadata["filename"].str.replace(
        r"_[12]\.fastq\.gz$", "", regex=True
    )
    r1_metadata = (
        metadata[metadata["filename"].str.endswith("_1.fastq.gz")]
        .set_index("sample_stem")
    )
    enriched = samplesheet.merge(
        r1_metadata, left_on="sample", right_index=True, how="left",
    )
    preferred = [c for c in [
        "sample", "fastq_1", "fastq_2",
        "SAMPLE ID", "PATHOGEN/ORGANISM NAME (OR METAGENOMIC)",
        "BIOSPECIMEN TYPE", "DATE COLLECTED",
        "SEQUENCING INSTRUMENT MAKE AND MODEL",
        "SEQUENCING LAB (ORIGINATING LAB)", "SOURCE LOCATION STATE",
    ] if c in enriched.columns]
    print(f"Enriched view ({len(enriched)} samples):\n")
    print(enriched[preferred].to_string(index=False))
except FileNotFoundError:
    print(f"No _apgap_metadata.csv found in {DATASET_URI} — skipping enrichment.")
    print("(The samplesheet.csv we wrote above still works for launching.)")

## Switching from test to real data (Mode 2)

The cell below launches viralrecon against the samplesheet we built from your dataset bucket. It's commented out on purpose. Real data means **real time and real money** — a full run takes hours and costs tens to hundreds of dollars depending on volume.

### Un-comment checklist

Go through this checklist top-to-bottom before uncommenting the real-data cell:

1. **The test run above completed successfully** (Mode 1). If the test failed, fix that first.
2. **The samplesheet builder cell ran successfully** and produced a `samplesheet.csv` with the samples you actually want to run.
3. **You've reviewed the samplesheet** and confirmed the sample names and FASTQ paths look right.
4. **Your project's budget covers this run.** Rough estimate: a few dollars per sample for viralrecon at typical viral genome scale, more for larger data. Check with your lead if you're not sure.
5. **You know which `--genome` reference is correct** for your pathogen. The commented cell defaults to SARS-CoV-2's `MN908947.3`; for Influenza A H1N1 you'd use `FJ966082.1`, or pass `--fasta` + `--gff` for a custom reference. See the [viralrecon usage docs](https://nf-co.re/viralrecon/usage) for reference options.

### How to run it

In the next cell, uncomment every line (select all lines with `Ctrl+A` inside the cell, then `Ctrl+/` in JupyterLab), review the params one more time, then run it. Nextflow will submit each pipeline step as a GCP Batch job and stream progress inline.

Watch on GCP: https://console.cloud.google.com/batch/jobs

In [ ]:
# import subprocess
# subprocess.run([
#     "nextflow", "-ansi-log", "false",   # cleaner Jupyter output; see launch cell above
#     "run", PIPELINE_REPO,
#     "-r", PIPELINE_REVISION,
#     "-profile", "gcb",       # NOTE: just "gcb", not "test,gcb"
#     "-c", "nextflow.config",
#     "-resume",
#     "--input", "samplesheet.csv",
#     "--outdir", RESULTS_BUCKET.rstrip("/") + "/viralrecon-real-results",  # no trailing /
#     "--platform", "illumina",
#     "--protocol", "metagenomic",  # or 'amplicon' if you used a primer scheme
#     "--genome", "MN908947.3",     # SARS-CoV-2 default. For Influenza A H1N1
#                                   # try "FJ966082.1" or pass a custom --fasta
#                                   # + --gff pair. See nf-co.re/viralrecon docs.
# ], check=True)

## Cleaning up

After a successful run you can delete the intermediate work
directory to save on storage. The final outputs in `--outdir` are
untouched.

This cell is commented out too. Uncomment when you're sure you won't
want to `-resume` from this run later. Deleting the work directory
destroys the cache `-resume` reads from, so any re-run after cleanup
starts fresh from step 1.

In [ ]:
# import subprocess
# subprocess.run(["gsutil", "-m", "rm", "-r", WORK_BUCKET], check=True)
# print(f"Deleted work directory {WORK_BUCKET}")

## Watching runs on GCP

- GCP Batch jobs console: https://console.cloud.google.com/batch/jobs
  (pick your project from the top-bar picker)
- Cloud Storage browser: https://console.cloud.google.com/storage
- Seqera workspace (if your project has one) is linked from the
  project detail page in the APGAP web app

## Where to go next

- Build a custom samplesheet with different groupings or extra
  metadata: see the [viralrecon usage docs](https://nf-co.re/viralrecon/usage)
- Try a different pipeline: browse the [nf-core catalog](https://nf-co.re/pipelines)
- Prepare more input data: [04-download-from-public-repos.ipynb](04-download-from-public-repos.ipynb)
- Back to the [getting-started overview](01-getting-started.ipynb)